# 42 - Re-audit of `main.tex`'s Artefact 2/3 Raw-Trajectory FIM Claims

`notebooks/33_fim_cross_system_validation.ipynb` computed the raw-trajectory FIM numbers
`main.tex` cites for Artefact 2 (`(alpha, beta_r)` at W11, claimed to collapse from ~0.85-0.99
to ~0.00-0.03) and Artefact 3 (`(alpha, z_A0_eff)` at W2, claimed off-diagonal
-0.89 -> -0.07, condition number 4.36e3 -> 5.48; `(beta_r, z_A0_eff)` at W5, condition number
41.4 -> 5.24) -- **using `fim_utils.offdiag_ratio`, the same raw/uninverted information-matrix
metric that `notebooks/39-41` found gives the wrong sign relative to an actual trained
posterior.** This notebook recomputes all three claims at their exact cited operating points
(`W2`, `W5`, `W11`, from `main.tex` Table `tab:si_wu_scenarios`) using both the original raw
metric (to confirm reproduction of the published numbers) and the corrected
`fim_utils.cov_offdiag_corr` (added 2026-09-01), side by side.

**Why this matters more than finalising the manuscript framing right now:** if Artefact 3's
claimed collapse does not survive the fix, `main.tex`'s core conclusion for System II --
"these are representation artefacts... not plant-level non-identifiabilities" -- is wrong for
Artefact 3, and needs to be rewritten as a confirmed genuine limit (consistent with
`notebooks/39-41`'s independent findings at nearby operating points: `nb40`'s `P2_alpha_only`,
`theta=[0.82,1,1,1,0.90]`, is essentially the same point as W2, `theta=[0.85,1,1,1,0.90]`, and
found `+0.63` raw-trajectory correlation there, not near-zero).

Deterministic-simulator only, no training -- should run in under a minute.

In [1]:
import pathlib
import numpy as np
import pandas as pd
import jax.numpy as jnp

cwd = pathlib.Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
import sys
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'scripts'))

from cstr_sbi.recycle.physics import (
    PARAM_NAMES, NOMINAL_CTRL_SB, NOMINAL_INLET,
    simulate_trajectory_explicit_jit, extract_observations_explicit,
)
from cstr_sbi.recycle.summaries import compute_summaries, RAW_INDEX, SB_CHANNELS
from cstr_sbi.recycle.simulator import nominal_warm_start
from fim_utils import compute_fim, offdiag_ratio, cov_offdiag_corr, crb_report

RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)

STRUCTURE = 'S-B'
NOISE_PCT = 0.003
EPS_FIM = np.array([0.02, 0.02, 0.02, 0.02, 0.01])
Y0_FIM = np.asarray(nominal_warm_start('S-B'))
ALPHA_IDX, BETA_R_IDX, ETA_COL_IDX, XI_REB_IDX, Z_A0_IDX = range(5)

# Exact scenario thetas from main.tex Table tab:si_wu_scenarios / src/cstr_sbi/recycle/scenarios.py
W2 = np.array([0.85, 1.00, 1.00, 1.00, 0.90], dtype=np.float64)   # Cat. decay mild -- Artefact 3 (alpha, z_A0_eff)
W5 = np.array([1.00, 0.80, 1.00, 1.00, 0.90], dtype=np.float64)   # Jacket fouling mild -- Artefact 3 (beta_r, z_A0_eff)
W11 = np.array([0.80, 0.80, 1.00, 1.00, 0.90], dtype=np.float64)  # Cat. + jacket -- Artefact 2 (alpha, beta_r)

print('W2  (Artefact 3, alpha/z_A0_eff): ', W2.tolist())
print('W5  (Artefact 3, beta_r/z_A0_eff):', W5.tolist())
print('W11 (Artefact 2, alpha/beta_r):   ', W11.tolist())

W2  (Artefact 3, alpha/z_A0_eff):  [0.85, 1.0, 1.0, 1.0, 0.9]
W5  (Artefact 3, beta_r/z_A0_eff): [1.0, 0.8, 1.0, 1.0, 0.9]
W11 (Artefact 2, alpha/beta_r):    [0.8, 0.8, 1.0, 1.0, 0.9]


In [2]:
def sim_raw_full(theta_np, seed=None, noise_pct=NOISE_PCT):
    th = jnp.array(theta_np, dtype=jnp.float32)
    ts, ys = simulate_trajectory_explicit_jit(th, NOMINAL_INLET, NOMINAL_CTRL_SB, jnp.array(Y0_FIM),
                                               t_final=2.0, n_save=120, rtol=1e-3, atol=1e-5)
    raw = np.asarray(extract_observations_explicit(ys, th, NOMINAL_CTRL_SB))
    t_h = np.asarray(ts)
    if seed is not None:
        rng = np.random.default_rng(seed)
        scale = np.maximum(np.max(np.abs(raw), axis=0), 1e-6)
        raw = raw + rng.normal(0, noise_pct * scale, raw.shape)
    return raw, t_h


def feat_summary_fim(theta, seed):
    raw, t_h = sim_raw_full(theta, seed)
    return compute_summaries(raw, 'S-B', t_h)


def feat_raw_fim(theta, seed, channels=SB_CHANNELS):
    raw, t_h = sim_raw_full(theta, seed)
    idx = [RAW_INDEX[c] for c in channels]
    return raw[:, idx].reshape(-1)


AUDIT_POINTS = [
    dict(name='W2 (Artefact 3: alpha/z_A0_eff)', theta=W2, idx_a=ALPHA_IDX, idx_b=Z_A0_IDX,
         claimed='off-diag -0.89 (66D) -> -0.07 (raw); cond 4.36e3 -> 5.48'),
    dict(name='W5 (Artefact 3: beta_r/z_A0_eff)', theta=W5, idx_a=BETA_R_IDX, idx_b=Z_A0_IDX,
         claimed='cond 41.4 (66D) -> 5.24 (raw) [off-diag not stated numerically in main.tex]'),
    dict(name='W11 (Artefact 2: alpha/beta_r)', theta=W11, idx_a=ALPHA_IDX, idx_b=BETA_R_IDX,
         claimed='posterior corr ~0.998; off-diag collapses ~0.00-0.03 under raw trajectory (nb29b)'),
]

audit_rows = []
for point in AUDIT_POINTS:
    theta_pt, ia, ib = point['theta'], point['idx_a'], point['idx_b']
    print(f"\n=== {point['name']} ===")
    print(f"    main.tex claims: {point['claimed']}")
    for repr_label, fn in [('compute_summaries (66-D)', feat_summary_fim), ('raw trajectory (1080-D)', feat_raw_fim)]:
        for off in [0, 500, 999]:
            FIM_rep = compute_fim(fn, theta_pt, EPS_FIM, seed_offset=off)
            raw_od, ratio = offdiag_ratio(FIM_rep, ia, ib)
            corr = cov_offdiag_corr(FIM_rep, ia, ib)
            crb = crb_report(FIM_rep)
            print(f'    {repr_label:26s} seed_offset={off:4d}: raw_off_diag={raw_od:+.3f}'
                  f'  corr(cov)={corr:+.3f}  cond={crb["cond"]:.3e}')
            audit_rows.append({'point': point['name'], 'representation': repr_label, 'seed_offset': off,
                                'raw_off_diag': raw_od, 'corr_cov': corr, 'cond': crb['cond']})

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(RESULTS / '42_wu2003_manuscript_fim_reaudit.csv', index=False)
display(audit_df.groupby(['point', 'representation'])[['raw_off_diag', 'corr_cov', 'cond']].mean().round(4))


=== W2 (Artefact 3: alpha/z_A0_eff) ===
    main.tex claims: off-diag -0.89 (66D) -> -0.07 (raw); cond 4.36e3 -> 5.48


    compute_summaries (66-D)   seed_offset=   0: raw_off_diag=-0.886  corr(cov)=+0.794  cond=2.045e+03


    compute_summaries (66-D)   seed_offset= 500: raw_off_diag=-0.877  corr(cov)=+0.758  cond=2.023e+03


    compute_summaries (66-D)   seed_offset= 999: raw_off_diag=-0.848  corr(cov)=+0.676  cond=3.329e+03


    raw trajectory (1080-D)    seed_offset=   0: raw_off_diag=-0.801  corr(cov)=+0.593  cond=1.361e+02


    raw trajectory (1080-D)    seed_offset= 500: raw_off_diag=-0.792  corr(cov)=+0.580  cond=1.247e+02


... (10 outputs left out)

    raw trajectory (1080-D)    seed_offset=   0: raw_off_diag=-0.034  corr(cov)=+0.049  cond=1.099e+02


    raw trajectory (1080-D)    seed_offset= 500: raw_off_diag=-0.094  corr(cov)=+0.132  cond=1.016e+02


    raw trajectory (1080-D)    seed_offset= 999: raw_off_diag=-0.077  corr(cov)=+0.076  cond=9.966e+01


raw_off_diag  \
point                            representation                           
W11 (Artefact 2: alpha/beta_r)   compute_summaries (66-D)       -0.2081   
                                 raw trajectory (1080-D)        -0.0683   
W2 (Artefact 3: alpha/z_A0_eff)  compute_summaries (66-D)       -0.8704   
                                 raw trajectory (1080-D)        -0.7933   
W5 (Artefact 3: beta_r/z_A0_eff) compute_summaries (66-D)        0.1389   
                                 raw trajectory (1080-D)         0.0220   

                                                           corr_cov       cond  
point                            representation                                 
W11 (Artefact 2: alpha/beta_r)   compute_summaries (66-D)    0.1942  1346.2894  
                                 raw trajectory (1080-D)     0.0857   103.7017  
W2 (Artefact 3: alpha/z_A0_eff)  compute_summaries (66-D)    0.7426  2465.4368  
                                 raw trajectory (1080-D)     0.5800   128.8887  
W5 (Artefact 3: beta_r/z_A0_eff) compute_summaries (66-D)   -0.3114   157.0367  
                                 raw trajectory (1080-D)    -0.0485    44.0590

## Verdict

**The exact cited numbers (-0.07, 4.36e3->5.48, 41.4->5.24) could not be reproduced** with
this notebook's standard methodology (full 9-channel `SB_CHANNELS` raw trajectory, fixed
nominal warm start, `EPS_FIM=[0.02,0.02,0.02,0.02,0.01]` -- identical to `nb29b`/`nb39`/`nb40`/
`nb41`), and a repo-wide search found these literal numbers in no tracked notebook or script
-- their provenance is untraceable as things stand. That is itself worth flagging
independent of the sign-bug question.

**But the qualitative, decision-relevant picture splits cleanly across the two pairs, and is
consistent with everything else this project has found:**

- **W2 `(alpha, z_A0_eff)`: does NOT collapse.** Raw-trajectory `corr(cov)` stays large
  (`+0.57` to `+0.79`), barely smaller than the 66-D value (`+0.68` to `+0.79`). This is a
  5th independent confirmation (after `nb39`, `nb40`'s `P2` at nearly the same theta, and
  `nb41`) that this specific coupling is a genuine, representation-robust limit, not a fixable
  artefact. **`main.tex`'s Artefact 3 claim for `(alpha, z_A0_eff)` needs to be corrected**,
  not just its numbers -- the qualitative conclusion is wrong, not only the cited values.
- **W5 `(beta_r, z_A0_eff)`: mostly does collapse.** `corr(cov)` drops from moderate
  (`-0.28` to `-0.34`) at 66-D to near-zero (`-0.04` to `-0.05`) at raw trajectory -- a real,
  substantial reduction, qualitatively consistent with `main.tex`'s claim even though the
  exact condition-number magnitudes don't match. **This is good news for the manuscript's
  scope**: `beta_r` (unlike `alpha`) does not appear to share a genuine identifiability limit
  with `z_A0_eff` -- physically sensible, since `beta_r` (jacket heat transfer) has no direct
  stoichiometric link to feed composition the way `alpha` (reaction rate) and `z_A0_eff`
  (feed composition) do.
- **W11 `(alpha, beta_r)`, Artefact 2 (currently reported as fully resolved and operationally
  benign): does not reproduce the claimed near-total collapse using this notebook's
  methodology at this exact point** -- 66-D `corr(cov)` here is only `+0.05` to `+0.28`, far
  below the `+0.901`/`+0.998` figures reported elsewhere in the manuscript/`nb29b` for this
  pair (measured at nominal and via the trained posterior, not at W11's FIM directly). This
  is **not evidence Artefact 2 is wrong** -- different operating points can legitimately give
  different coupling strength, and this pair's own literature in this project already
  includes real point-dependence -- but it means Artefact 2 has **not** yet received the same
  dedicated, multi-point, corrected-metric scrutiny that Artefact 3 just did, and should
  before being treated as settled.

**Recommendation:** treat Artefact 3's `(alpha, z_A0_eff)` finding as confirmed and rewrite
`main.tex` accordingly (see chat response for suggested framing). Do not touch Artefact 2's
text yet -- schedule a dedicated `nb39`/`40`/`41`-style re-validation (multiple points,
grid-scan, raw trajectory, actual posterior) before deciding whether it also needs revision.